# Oxygen sampling–mapping trend differences

Produces the 1965–2021 boxplot and map figure and the 1993–2021 supplementary boxplots.


In [ ]:
"""Set up the 1965--2021 boxplot/map and 1993--2021 supplementary boxplots."""

import os
import pickle
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-db9274")

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter, MultipleLocator
import numpy as np


SOURCE_DIR = Path("/scratch/gpfs/LRGROUP/db9274/Variability_trends_EC/Paper_figures")
CACHE_FILE = SOURCE_DIR / (
    "cache_sampling_mapping_trends_different_periods/"
    "global_trend_sampling_mapping_samples_full_upper_below_1965_2021_"
    "6_products_olivelli_v3.pkl"
)
OUTPUT_DIR = Path.cwd() / "Figures"
BOX_OUTPUT = OUTPUT_DIR / "oxygen_sampling_mapping_difference_trends_two_periods_boxplots.pdf"

PERIODS = ["1965–2021", "1993–2021"]
DEPTHS = ["Full water column", "Upper 2000 m", "Below 2000 m"]
PRODUCTS = [
    "observation-Gouretski(2024)",
    "observation-Ito(2024; NN)",
    "observation-Ito(2024; RF)",
    "observation-Ito(2022; 5 year)",
    "observation-Roach and Bindoff",
    "observation-Olivelli(2026)",
]
LABELS = {
    "observation-Gouretski(2024)": "Gouretski et al. (2024)",
    "observation-Ito(2024; NN)": "Ito et al. (2024): Neural network",
    "observation-Ito(2024; RF)": "Ito et al. (2024): Random forest",
    "observation-Ito(2022; 5 year)": "Ito (2022)",
    "observation-Roach and Bindoff": "Roach & Bindoff (2023)",
    "observation-Olivelli(2026)": "Olivelli et al. (2026)",
}
COLORS = {
    "observation-Gouretski(2024)": "#CC79A7",
    "observation-Ito(2024; NN)": "#009E73",
    "observation-Ito(2024; RF)": "#D55E00",
    "observation-Ito(2022; 5 year)": "#0072B2",
    "observation-Roach and Bindoff": "#E69F00",
    "observation-Olivelli(2026)": "#7B2CBF",
}
XLABEL = "Subsampling-reconstruction error ($\\mu$mol kg$^{-1}$ decade$^{-1}$)"

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "figure.dpi": 130,
    "savefig.dpi": 300,
})



def symmetric_xlim(samples):
    selected = samples[samples["period"].isin(PERIODS)]
    values = selected["sampling_mapping_difference"].to_numpy(float)
    values = values[np.isfinite(values)]
    lo, hi = np.percentile(values, [1, 99])
    extent = 1.2 * max(abs(lo), abs(hi))
    return -extent, extent


def add_percent_axis(ax, data):
    full = data.drop_duplicates(["model", "ensemble"])["fully_sampled_trend"].to_numpy(float)
    reference = abs(np.nanmedian(full))
    if np.isfinite(reference) and reference > 0:
        sec = ax.secondary_xaxis(
            "top",
            functions=(lambda x: 100 * x / reference, lambda pct: pct * reference / 100),
        )
        sec.xaxis.set_major_formatter(FuncFormatter(lambda value, pos: f"{value:g}%"))
        sec.tick_params(axis="x", labelsize=10, pad=1.5)


def panel_data(samples, period, depth):
    return samples[(samples["period"] == period) & (samples["depth"] == depth)]



In [2]:

# Requested boxplot + six-method mean sampling–mapping maps (1965–2021 only).
import xarray as xr
from matplotlib.colors import TwoSlopeNorm, ListedColormap
from matplotlib.transforms import Bbox
from cartopy import crs as ccrs
import cartopy.feature as cfeature
# Keep the projection seam in the Atlantic rather than cutting through Africa.
MAP_CENTRAL_LONGITUDE = 200
SEAM_INTERPOLATION_LONGITUDES = (71.5, 72.5, 73.5, 74.5)
MAX_SEAM_INTERPOLATION_GAP_DEGREES = 80.0

MAP_FILE = Path('/scratch/gpfs/LRGROUP/db9274/Subsampled_o2_variability/Cause_model_obs_differences/Figures/all_models_multimodel_mean_mapped_minus_full_deoxygenation_difference_maps_1965_2021.nc')
FULL_MAP_CACHE = Path.cwd() / 'Figures/multimethod_mean_full_column_1965_2021.nc'

def _layer_edges(z):
    e=np.empty(len(z)+1); e[1:-1]=(z[:-1]+z[1:])/2; e[0]=max(0,z[0]-(z[1]-z[0])/2); e[-1]=z[-1]+(z[-1]-z[-2])/2; return e

def multimethod_maps():
    if not FULL_MAP_CACHE.exists():
        raise FileNotFoundError(f'Run build_multimethod_mean_full_column.ipynb first: {FULL_MAP_CACHE}')
    with xr.open_dataset(FULL_MAP_CACHE) as ds:
        methods=ds.product.values.tolist()
        full=ds.difference.values
        olivelli_upper=ds.olivelli_upper.values
        olivelli_lower=ds.olivelli_below.values
        full_lat=ds.y.values
        full_lon=((ds.x.values + 180) % 360) - 180
        full_order=np.argsort(full_lon); full=full[:,full_order]; olivelli_upper=olivelli_upper[:,full_order]; olivelli_lower=olivelli_lower[:,full_order]
    with xr.open_dataset(MAP_FILE) as ds:
        cached_methods=[method for method in methods if method != 'Olivelli et al. (2026)']
        if len(methods) != 6 or len(cached_methods) != 5 or set(cached_methods) != set(ds.product.values.tolist()):
            raise ValueError('Expected all six methods: five cached maps plus Olivelli')
        layers=ds.difference.sel(product=cached_methods)
        upper=layers.sel(depth='upper_2000_m').values
        lower=layers.sel(depth='below_2000_m').values
        lat,lon=ds.y.values,ds.x.values
        lon=((lon + 180) % 360) - 180
        order=np.argsort(lon); lon=lon[order]
        upper=np.concatenate([upper[:,:,order],olivelli_upper[None,:,:]],axis=0).mean(axis=0)
        lower=np.concatenate([lower[:,:,order],olivelli_lower[None,:,:]],axis=0).mean(axis=0)
        if not (np.allclose(lat,full_lat) and np.allclose(lon,full_lon[full_order])):
            raise ValueError('Full-water-column cache grid does not match upper/deep cache')
    return lat,lon,[full,upper,lower]
def _prepare_longitude_for_plot(field, lon):
    # Match the periodic native-column interpolation used by the source map figure.
    result=np.array(field,copy=True); x=np.asarray(lon,dtype=float); dx=float(np.nanmedian(np.diff(x)))
    targets=[]
    for target in SEAM_INTERPOLATION_LONGITUDES:
        while target < float(x.min()): target += 360.0
        while target > float(x.max()) + dx: target -= 360.0
        targets.append(target)
    native=sorted(float(x[np.argmin(np.abs(x-target))]) for target in targets if np.any(np.isclose(x,target)))
    groups=[]
    for target in native:
        if not groups or not np.isclose(target-groups[-1][-1],dx): groups.append([target])
        else: groups[-1].append(target)
    for group in groups:
        indices=[int(np.argmin(np.abs(x-target))) for target in group]; start=min(indices); end=max(indices); nx=len(x)
        for row in result:
            search=row.copy(); search[indices]=np.nan; left=right=None
            for step in range(1,nx):
                idx=(start-step)%nx
                if np.isfinite(search[idx]): left=idx; break
            for step in range(1,nx):
                idx=(end+step)%nx
                if np.isfinite(search[idx]): right=idx; break
            if left is None or right is None: continue
            left_x=float(x[left]); right_x=float(x[right])+(360.0 if right<=left else 0.0); span=right_x-left_x
            if span<=0 or span>MAX_SEAM_INTERPOLATION_GAP_DEGREES: continue
            for idx in indices:
                if np.isfinite(row[idx]): continue
                target=float(x[idx])+(360.0 if idx<=left else 0.0); weight=(target-left_x)/span
                row[idx]=row[left]+weight*(row[right]-row[left])
    return np.concatenate([result,result[:,:1]],axis=1),np.r_[x,x[-1]+dx]


def make_requested_box_map_figure(samples):
    fig=plt.figure(figsize=(11,7.5))
    gs=fig.add_gridspec(2,3,left=.17,right=.985,top=.91,bottom=.21,wspace=.22,hspace=.28,height_ratios=[1.4,1])
    period='1965–2021'
    xlim=symmetric_xlim(samples.loc[samples['period'] == period])
    column_titles=['Full-water column','Upper 2000 m','Below 2000 m']
    labels=['Gouretski et al.\n(2024)','Ito et al. (2024)\nNeural network','Ito et al. (2024)\nRandom forest','Ito (2022)','Roach & Bindoff\n(2023)','Olivelli et al.\n(2026)','Combined']; colors=[COLORS[p] for p in PRODUCTS]+['black']; pos=np.arange(len(labels),0,-1)
    for col,(depth,column_title) in enumerate(zip(DEPTHS,column_titles)):
        ax=fig.add_subplot(gs[0,col]); data=panel_data(samples,period,depth); groups=[data.loc[data["product"]==p,'sampling_mapping_difference'].dropna().to_numpy() for p in PRODUCTS]+[data.sampling_mapping_difference.dropna().to_numpy()]
        full=data.drop_duplicates(['model','ensemble'])['fully_sampled_trend'].to_numpy(float)
        reference=abs(np.nanmedian(full))
        if not np.isfinite(reference) or reference <= 0: raise ValueError(f'Invalid fully sampled trend for {depth}')
        regions=[(xlim[0],-reference,'#DCE9F5','Switches\ndeoxygen-\nnation sign'),(-reference,0,'#F5EAD9','Weakens\ndeoxygen-\nation'),(0,xlim[1],'#E1F0E5','Strengthens\ndeoxygenation')]
        for left,right,color,description in regions:
            ax.axvspan(left,right,facecolor=color,zorder=0)
            if col == 0: ax.text((left+right)/2,-.25,description,ha='center',va='center',fontsize=7.5,linespacing=.9)
        r=ax.boxplot(groups,vert=False,positions=pos,widths=.58,patch_artist=True,showfliers=True,medianprops={'color':'white','linewidth':1.4},flierprops={'marker':'o','markersize':2.8,'alpha':.55})
        for patch,color in zip(r['boxes'],colors): patch.set(facecolor=color,edgecolor=color,alpha=.5)
        ax.set_yticks(pos); ax.set_yticklabels(labels if col == 0 else [],fontsize=8); ax.set_ylim(-1.0,len(labels)+.65); format_common_panel(ax,xlim,data,False); ax.tick_params(axis='x',labelsize=10); ax.tick_params(axis='y',length=4,direction='out',pad=3); ax.child_axes[0].tick_params(axis='x',labelsize=10); ax.set_title(f'{chr(97+col)}. {column_title}',fontweight='bold',pad=24)
    lat,lon,fields=multimethod_maps(); finite=np.concatenate([f[np.isfinite(f)] for f in fields]); lim=float(np.nanpercentile(np.abs(finite),99)); norm=TwoSlopeNorm(vmin=-lim,vcenter=0,vmax=lim); mesh=None
    map_axes=[]
    for col,(depth,column_title,field) in enumerate(zip(DEPTHS,column_titles,fields)):
        ax=fig.add_subplot(gs[1,col],projection=ccrs.Robinson(central_longitude=MAP_CENTRAL_LONGITUDE))
        cyclic_field,cyclic_lon=_prepare_longitude_for_plot(field,lon)
        invalid=np.where(np.isfinite(cyclic_field),np.nan,1.0)
        ax.pcolormesh(cyclic_lon,lat,invalid,transform=ccrs.PlateCarree(),cmap=ListedColormap(['.86']),vmin=0,vmax=1,shading='auto',rasterized=True,zorder=1)
        mesh=ax.pcolormesh(cyclic_lon,lat,cyclic_field,transform=ccrs.PlateCarree(),cmap='bwr',norm=norm,shading='auto',rasterized=True,zorder=2)
        ax.set_global(); ax.add_feature(cfeature.LAND,facecolor='.86',edgecolor='none',zorder=3); ax.coastlines(linewidth=.45,color='.15',zorder=4); ax.gridlines(linewidth=.25,color='.55',alpha=.4); ax.spines['geo'].set_edgecolor('.15'); ax.spines['geo'].set_linewidth(.6); map_axes.append(ax)
        ax.set_title(f'{chr(100+col)}. {column_title}',fontweight='bold',pad=4)
    fig.text(.57,.51,'Subsampling-reconstruction error (µmol kg⁻¹ decade⁻¹)',va='center',ha='center',fontsize=10)
    fig.text(.025,.73,'Reconstruction frameworks',rotation=90,va='center',ha='center',fontweight='bold')
    fig.text(.025,.36,'Six-framework mean',rotation=90,va='center',ha='center',fontweight='bold')
    cax=fig.add_axes([.37,.14,.34,.026]); cb=fig.colorbar(mesh,cax=cax,orientation='horizontal',extend='both'); cb.ax.tick_params(labelsize=10); cb.set_label('Subsampling-reconstruction error (µmol kg⁻¹ decade⁻¹)',fontsize=10)
    for spine in cax.spines.values():
        spine.set_visible(True); spine.set_linewidth(.8); spine.set_edgecolor('black')
    cax.text(.23,1.55,'Weaker loss relative to\nfully sampled fields',transform=cax.transAxes,ha='center',va='bottom',fontsize=10)
    cax.text(.77,1.55,'Stronger loss relative to\nfully sampled fields',transform=cax.transAxes,ha='center',va='bottom',fontsize=10)
    fig.savefig(BOX_OUTPUT,bbox_inches=Bbox.from_bounds(0,.3,fig.get_figwidth(),fig.get_figheight()-.3)); plt.close(fig)

with CACHE_FILE.open('rb') as h: _samples=pickle.load(h)
make_requested_box_map_figure(_samples)
print(f'Saved {BOX_OUTPUT}')


Saved /scratch/gpfs/LRGROUP/db9274/Variability_trends_EC/Paper_figures_p2/Figures/oxygen_sampling_mapping_difference_trends_two_periods_boxplots.pdf


In [3]:

# Supplementary figure: boxplots for 1993–2021 only.
SUPP_OUTPUT = OUTPUT_DIR / 'oxygen_sampling_mapping_difference_trends_boxplots_1993_2021_supplement.pdf'

def make_supplementary_boxplots(samples, xlim):
    fig, axes = plt.subplots(1, 3, figsize=(11.0, 4.2), sharex=True)
    fig.subplots_adjust(left=0.17, right=0.98, top=0.89, bottom=0.25, wspace=0.22)
    references = {depth: abs(np.nanmedian(panel_data(samples, '1993–2021', depth).drop_duplicates(['model', 'ensemble'])['fully_sampled_trend'].to_numpy(float))) for depth in DEPTHS}
    if not all(np.isfinite(value) and value > 0 for value in references.values()): raise ValueError('Invalid fully sampled trend')
    labels = ['Gouretski et al.\n(2024)', 'Ito et al. (2024)\nNeural network', 'Ito et al. (2024)\nRandom forest', 'Ito (2022)', 'Roach & Bindoff\n(2023)', 'Olivelli et al.\n(2026)', 'Combined']
    colors = [COLORS[p] for p in PRODUCTS] + ['black']
    positions = np.arange(len(labels), 0, -1)
    for ax, depth, letter in zip(axes, DEPTHS, ('a', 'b', 'c')):
        data = panel_data(samples, '1993–2021', depth)
        reference = references[depth]
        regions = [(xlim[0], -reference, '#DCE9F5', 'Switches\ndeoxygen-\nnation sign'), (-reference, 0, '#F5EAD9', 'Weakens\ndeoxygenation'), (0, xlim[1], '#E1F0E5', 'Strengthens\ndeoxygenation')]
        for index, (left, right, color, description) in enumerate(regions):
            visible_left, visible_right = max(left, xlim[0]), min(right, xlim[1])
            if visible_left >= visible_right: continue
            ax.axvspan(visible_left, visible_right, facecolor=color, zorder=0)
            if letter == 'a' and index > 0:
                ax.text((visible_left + visible_right) / 2, -.25, description, ha='center', va='center', fontsize=7.5, linespacing=.9)
        groups = [data.loc[data['product'] == p, 'sampling_mapping_difference'].dropna().to_numpy() for p in PRODUCTS]
        groups += [data['sampling_mapping_difference'].dropna().to_numpy()]
        result = ax.boxplot(groups, vert=False, positions=positions, widths=0.58,
                            patch_artist=True, showfliers=True,
                            medianprops={'color': 'white', 'linewidth': 1.4},
                            whiskerprops={'linewidth': 1}, capprops={'linewidth': 1},
                            flierprops={'marker': 'o', 'markersize': 2.8, 'alpha': 0.55})
        for patch, color in zip(result['boxes'], colors):
            patch.set(facecolor=color, edgecolor=color, alpha=0.5)
        ax.axvline(0, color='0.35', lw=0.9, ls='--')
        ax.set_xlim(xlim); ax.set_ylim(-1.0, len(labels) + 0.65)
        if letter == 'a':
            blue_midpoint = (xlim[0] - reference) / 2
            ax.annotate('Switches\ndeoxygenation sign', xy=(blue_midpoint, .5), xycoords='data', xytext=(.01, -.16), textcoords='axes fraction', ha='left', va='top', fontsize=7.5, annotation_clip=False, arrowprops=dict(arrowstyle='->', lw=.8, color='.25', relpos=(0, 1)))
        ax.xaxis.set_major_locator(MultipleLocator(0.2))
        ax.set_yticks(positions); ax.set_yticklabels(labels if letter == 'a' else [], fontsize=8)
        ax.tick_params(axis='y', length=4, direction='out', pad=3)
        ax.grid(axis='x', ls=':', alpha=0.25)
        ax.tick_params(axis='x', labelsize=10, pad=1.5)
        ax.tick_params(axis='y', labelsize=8.5, pad=1.5)
        ax.set_title(f'{letter}. {depth}', fontsize=11, fontweight='bold')
        add_percent_axis(ax, data)
    fig.supxlabel(XLABEL, fontsize=10, x=0.55, y=0.09)
    fig.savefig(SUPP_OUTPUT, bbox_inches='tight')
    plt.close(fig)

with CACHE_FILE.open('rb') as handle:
    _supp_samples = pickle.load(handle)
make_supplementary_boxplots(_supp_samples, symmetric_xlim(_supp_samples.loc[_supp_samples['period'] == '1965–2021']))
print(f'Saved {SUPP_OUTPUT}')


Saved /scratch/gpfs/LRGROUP/db9274/Variability_trends_EC/Paper_figures_p2/Figures/oxygen_sampling_mapping_difference_trends_boxplots_1993_2021_supplement.pdf
